# Fabric defect - Kaggle GPU runs (clean stratified split)
Settings: Accelerator = GPU (T4/P100), Internet = ON (pretrained weights).
Add the uploaded dataset (kaggle_upload_clean.zip) via 'Add Input'.
Split: splits_clean.json (StratifiedGroupKFold, fixed). Only the training seed changes.

In [ ]:
!nvidia-smi | head -n 12
import os, glob, shutil, zipfile
z = glob.glob('/kaggle/input/**/kaggle_upload_clean.zip', recursive=True)
if z:
    zipfile.ZipFile(z[0]).extractall('/kaggle/working/proj')
else:  # dataset uploaded as extracted folder
    root = os.path.dirname(glob.glob('/kaggle/input/**/train.py', recursive=True)[0])
    shutil.copytree(root, '/kaggle/working/proj', dirs_exist_ok=True)
%cd /kaggle/working/proj
!ls
import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

In [ ]:
# A) pretrained CNNs x strategies x 3 seeds, 224px. Separate out dir per run (summary.csv is keyed by model).
import subprocess
def run(model, seed, strategy='full', epochs=15, lr='1e-4', size=224):
    subprocess.run(['python','train.py','--model',model,'--data','data_clean','--splits','splits_clean.json',
                    '--stratified','--strategy',strategy,'--epochs',str(epochs),'--lr',lr,'--size',str(size),
                    '--bs','32','--workers','2','--seed',str(seed),
                    '--out',f'runs_clean/{strategy}_seed{seed}'], check=True)
for seed in (42, 43, 44):
    for m in ('resnet50','efficientnet_b0','convnext_tiny'):
        run(m, seed)
    for st in ('frozen','gradual','discriminative'):
        run('convnext_tiny', seed, st)

In [ ]:
# B) from-scratch models (not yet run on real data - check the first epochs before waiting for all).
# vision_mamba is a sequential pure-PyTorch scan: 128px keeps it tractable.
for seed in (42, 43, 44):
    run('vit', seed, epochs=30, lr='3e-4')
    run('hybrid', seed, epochs=30, lr='3e-4')
    run('vision_mamba', seed, epochs=30, lr='3e-4', size=128)

In [ ]:
import pandas as pd, glob, os
rows=[]
for f in glob.glob('runs_clean/*/summary.csv'):
    d=os.path.basename(os.path.dirname(f)); st,seed=d.split('_seed')
    rows.append(pd.read_csv(f).assign(strategy=st, seed=int(seed)))
df=pd.concat(rows)
print(df.groupby(['model','strategy'])[['test_acc','test_macro_f1','val_best_macro_f1']].agg(['mean','std']).round(4))
!zip -rq /kaggle/working/results.zip runs_clean -x '*.pt'